# 04 - Gold + cleanup

Task `Step_03_Gold_Cleanup` of the pipeline job (replaces `05_gold_merge` + `06_cleanup`). Each part logs to `ops.pipeline_runs` under its own task name.

**1. LinkedIn config** (`export_linkedin_config`, written first so the next scrape gets them even if a later step fails), in `/Volumes/<catalog>/ops/pipeline/config/`
* `linkedin_roles.json` - `{"generated_at", "profiles": [{"user_id", "roles", "cities"}]}`: the roles each app user picked (`ops.user_profile`; custom roles once `03_enrich` marked them active / mapped) and up to 3 preferred cities (`ops.ref_india_locations` city names, may be empty = India-wide)
* `linkedin_seen_ids.json` - `{"generated_at", "job_ids": [...]}`: LinkedIn job ids first seen in the last 3 days that silver already has a description for. The scraper still emits them (so `last_seen_at` refreshes) but skips the description fetch; the silver MERGE never overwrites a stored description with an empty one

**2. Gold MERGE** (`gold_merge`) into `gold.jobs` (SCD Type 1, one row per `job_key`)

| Rule | Behaviour |
|---|---|
| Source | `silver.jobs_clean` joined to `silver.jobs_enriched` on `job_key` **and** `description_hash`, `gold_eligible = true`, posted (else first seen) within `job_delete_after_days`, so deleted jobs are never re-inserted |
| Insert | New `job_key` -> tracking defaults `is_applied = false`, `is_hidden = false`, `application_status = 'not_applied'`, `is_active = true` |
| Update | Only when a content column changed (description, enrichment, role...); a sighting alone (`first_seen_at`, `last_seen_at`, `times_seen`) does not rewrite the row, but is carried along when content changes. **Tracking columns are never touched** |
| Not written | `ats_type`, `company_url`, `posted_days`, `category_score`, `enrichment_version` (nothing reads them; the columns stay) |

**3. Cleanup** (`cleanup`). `dry_run = true` reports what steps 3a and 3c would remove or change without doing it (the exports, the MERGE and the 3b skill stats still run; 3d is skipped).

| Step | When | Rule (defaults) |
|---|---|---|
| 3a. Expire gold jobs | every run | `is_active = false` for jobs **posted** more than `job_expire_after_days` (2) ago (posted date, else first seen). Scrapers only fetch fresh postings, so "last seen" can't tell if a job is still open |
| 3a. Delete gold jobs | every run | Expired jobs older than `job_delete_after_days` (2, >= expiry) **unless tracked**: kept if any user applied or set a status in `gold.user_job_state` (or the legacy `is_applied` / `application_status` columns say so). Hidden-only jobs are deleted. Jobs `03_enrich` excluded as outside the supported roles (`out_of_scope`) are deleted too, with the same exception. Orphan `gold.user_job_state` rows are removed |
| 3b. Skill stats | every run | `ops.skill_stats` rebuilt from active gold jobs: `lower(skill or skill group)`, `jobs`, `idf = ln(1 + active jobs / jobs)` (the API weights matched skills by it) |
| 3c. Retention | at most every `retention_interval_hours` (24) | Landing CSVs bronze has ingested, older than `landing_retention_days` (7; files never ingested are never touched); bronze rows older than `bronze_retention_days` (30) from runs silver processed; rejected rows (30 days); run log (90 days, longer than bronze: silver uses it to know which bronze runs it processed); silver / enriched rows older than `job_delete_after_days` that are not in gold; LLM log failures / legacy rows / superseded prompt versions / orphans; failed and orphan job embeddings (role embeddings are kept) |
| 3d. Delta maintenance | at most every `maintenance_interval_hours` (168) | `OPTIMIZE` + `VACUUM` on the pipeline tables |

Row counts come from the DML result (`num_affected_rows`); only a dry run counts first. Net effect: gold keeps jobs posted in the last 2 days plus every job any user applied to or tracks.

**4. Publish files for Supabase** (`publish_files`, also in a dry run) in `/Volumes/<catalog>/ops/pipeline/publish/<job_run_id>/`. The app reads Supabase only; `databricks/jobs/publish_to_supabase.py` (GitHub Actions, after this run) sends the files there, so Databricks never holds a Supabase credential.

| File | Content |
|---|---|
| `jobs-00000.jsonl.gz` ... | The full `gold.jobs` snapshot sorted by `job_key`, 2000 rows per file, the app columns only (timestamps UTC ISO, dates `YYYY-MM-DD`) plus `h` = sha256 of the row's canonical JSON, so the publisher sends only changed rows |
| `keys.json` | `[[job_key, h], ...]`: every job that should exist (Supabase deletes the others unless a user tracks them) |
| `skill_stats.json`, `role_similarity.json`, `refs.json`, `custom_statuses.json` | `ops.skill_stats` (after 3b), `ops.role_similarity`, `ops.ref_roles` / `ops.ref_skills` / `ops.ref_india_locations`, `ops.custom_roles` / `ops.custom_skills` |
| `manifest.json` | Written last (present = complete): run id, `snapshot_at`, the `user_data_exported_at` this run loaded, every file's sha256 + bytes, row counts |

Retention: the 3 newest complete directories are kept; older complete ones are deleted after 3 days, directories without a manifest (failed runs) after 1 day (a dry run only reports them).

**Job task environment:** no extra dependencies needed.

In [ ]:
%run ./_common

In [ ]:
from datetime import datetime, timedelta, timezone

from pyspark.sql import functions as F

dbutils.widgets.text("landing_path", "/Volumes/jobseeker/default/scrapes", "Landing volume path")
dbutils.widgets.text("landing_retention_days", "7", "Delete ingested landing files after N days")
dbutils.widgets.text("bronze_retention_days", "30", "Delete processed bronze rows after N days")
dbutils.widgets.text("rejected_retention_days", "30", "Delete rejected rows after N days")
dbutils.widgets.text("run_log_retention_days", "90", "Delete pipeline run log rows after N days")
dbutils.widgets.text("llm_failed_retention_days", "7", "Delete failed / legacy LLM log rows after N days")
# Only fresh jobs matter: anything posted more than 2 days ago that you did not apply to is removed
dbutils.widgets.text("job_expire_after_days", "2", "Expire gold jobs posted more than N days ago")
dbutils.widgets.text("job_delete_after_days", "2", "Delete untracked expired jobs posted more than N days ago")
dbutils.widgets.text("retention_interval_hours", "24", "Run the retention steps at most every N hours")
dbutils.widgets.text("maintenance_interval_hours", "168", "Run OPTIMIZE/VACUUM at most every N hours")
dbutils.widgets.dropdown("dry_run", "false", ["false", "true"], "Dry run (report the cleanup only)")


def positive_int(name):
    value = int(dbutils.widgets.get(name))
    if value < 1:
        raise ValueError(f"{name} must be >= 1, got {value}")
    return value


LANDING_PATH = dbutils.widgets.get("landing_path").rstrip("/")
LANDING_DAYS = positive_int("landing_retention_days")
BRONZE_DAYS = positive_int("bronze_retention_days")
REJECTED_DAYS = positive_int("rejected_retention_days")
RUN_LOG_DAYS = positive_int("run_log_retention_days")
LLM_FAILED_DAYS = positive_int("llm_failed_retention_days")
EXPIRE_DAYS = positive_int("job_expire_after_days")
DELETE_DAYS = positive_int("job_delete_after_days")
RETENTION_HOURS = positive_int("retention_interval_hours")
MAINTENANCE_HOURS = positive_int("maintenance_interval_hours")
DRY_RUN = dbutils.widgets.get("dry_run") == "true"

if not LANDING_PATH.startswith("/Volumes/"):
    raise ValueError(f"landing_path must be a Unity Catalog volume path: {LANDING_PATH}")
if BRONZE_DAYS <= LANDING_DAYS:
    raise ValueError("bronze_retention_days must be greater than landing_retention_days (landing cleanup checks bronze)")
if RUN_LOG_DAYS <= BRONZE_DAYS:
    raise ValueError("run_log_retention_days must be greater than bronze_retention_days (silver uses the run log)")
if DELETE_DAYS < EXPIRE_DAYS:
    raise ValueError("job_delete_after_days must be >= job_expire_after_days (only expired jobs are deleted)")

BRONZE_TABLE = table("bronze", "jobs_raw")
SILVER_TABLE = table("silver", "jobs_clean")
REJECTED_TABLE = table("silver", "jobs_rejected")
ENRICHED_TABLE = table("silver", "jobs_enriched")
GOLD_TABLE = table("gold", "jobs")
USER_STATE_TABLE = table("gold", "user_job_state")
RUNS_TABLE = table("ops", "pipeline_runs")
LLM_LOG_TABLE = table("ops", "enrichment_llm_log")
EMBEDDING_CACHE_TABLE = table("ops", "embedding_cache")
SKILL_STATS_TABLE = table("ops", "skill_stats")
REF_ROLES_TABLE = table("ops", "ref_roles")
CUSTOM_ROLES_TABLE = table("ops", "custom_roles")
INDIA_REF_TABLE = table("ops", "ref_india_locations")
PROFILE_TABLE = table("ops", "user_profile")
MAINTAINED_TABLES = [BRONZE_TABLE, SILVER_TABLE, REJECTED_TABLE, ENRICHED_TABLE, GOLD_TABLE, USER_STATE_TABLE,
                     RUNS_TABLE, LLM_LOG_TABLE, EMBEDDING_CACHE_TABLE, SKILL_STATS_TABLE]

CONFIG_DIR = f"{PIPELINE_VOLUME}/config"
LINKEDIN_ROLES_FILE = f"{CONFIG_DIR}/linkedin_roles.json"
LINKEDIN_SEEN_IDS_FILE = f"{CONFIG_DIR}/linkedin_seen_ids.json"
MAX_CITIES = 3          # preferred cities per user (the profile editor allows 3)
SEEN_IDS_DAYS = 3       # LinkedIn ids first seen this recently are exported as already fetched

# Posting age used for expiry: posted date, else the day the job was first scraped
JOB_AGE_DATE = "coalesce(posted_date, CAST(first_seen_at AS DATE))"
# Gold jobs no user tracks: not applied and no status in gold.user_job_state (or the legacy gold.jobs columns)
UNTRACKED_JOB = (
    "NOT coalesce(is_applied, false) AND coalesce(application_status, 'not_applied') = 'not_applied' "
    f"AND NOT EXISTS (SELECT 1 FROM {USER_STATE_TABLE} AS s WHERE s.job_key = t.job_key "
    "AND (coalesce(s.is_applied, false) OR coalesce(s.application_status, 'not_applied') <> 'not_applied'))"
)

print(f"dry_run={DRY_RUN} landing={LANDING_DAYS}d bronze={BRONZE_DAYS}d rejected={REJECTED_DAYS}d runs={RUN_LOG_DAYS}d "
      f"llm_failed={LLM_FAILED_DAYS}d expire={EXPIRE_DAYS}d delete={DELETE_DAYS}d "
      f"retention={RETENTION_HOURS}h maintenance={MAINTENANCE_HOURS}h")

## Helpers

In [ ]:
report = {}

# The target table is aliased `t` in every statement below, so predicates can reference it (see missing_from).


def dml_rows(result, key="num_affected_rows"):
    """Row count reported by a Delta DML statement (no separate count() action); None if it reports none."""
    row = result.first()
    if row is None or key not in row.asDict():
        return None
    return int(row[key] or 0)


def count_where(table_name, predicate):
    return int(spark.sql(f"SELECT count(*) AS n FROM {table_name} AS t WHERE {predicate}").first()["n"])


def delete_where(step, table_name, predicate):
    """Deletes rows (a dry run only counts them) and records the count under report[step]."""
    if DRY_RUN:
        rows = count_where(table_name, predicate)
    else:
        rows = dml_rows(spark.sql(f"DELETE FROM {table_name} AS t WHERE {predicate}")) or 0
    report[step] = rows
    print(f"{'[DRY RUN] would delete' if DRY_RUN else 'deleted'} {rows:>6} rows  {step}")
    return rows


def update_where(step, table_name, assignments, predicate):
    if DRY_RUN:
        rows = count_where(table_name, predicate)
    else:
        rows = dml_rows(spark.sql(f"UPDATE {table_name} AS t SET {assignments} WHERE {predicate}")) or 0
    report[step] = rows
    print(f"{'[DRY RUN] would update' if DRY_RUN else 'updated'} {rows:>6} rows  {step}")
    return rows


def missing_from(table_name, column="job_key"):
    """Rows of the target whose `column` matches no job_key in table_name (orphans).
    NOT EXISTS rather than NOT IN: a plain anti join, and a NULL key in table_name cannot silently block the cleanup."""
    return f"NOT EXISTS (SELECT 1 FROM {table_name} AS ref WHERE ref.job_key = t.{column})"


def sql_literal(value):
    return "'" + str(value).replace("\\", "\\\\").replace("'", "\\'") + "'"


def put_json(path, payload, indent=None):
    dbutils.fs.put(path, json.dumps(payload, indent=indent, ensure_ascii=False), overwrite=True)

## 1. LinkedIn config exports
Written before the MERGE. The LinkedIn scraper (GitHub Actions) downloads both files: one search per distinct role x preferred city (India-wide for users without cities), skipping the description fetch for ids in `linkedin_seen_ids.json`. Fit scores are computed per user by the API at query time, so nothing is scored here.

In [ ]:
with task_run("export_linkedin_config") as metrics:
    valid_roles = {row["role_title"] for row in spark.table(REF_ROLES_TABLE).where("in_scope").select("role_title").collect()}
    # Roles users typed (checked by 03_enrich): active ones are searched as typed, mapped ones as the role they duplicate
    custom = {row["role_title"].lower(): row for row in spark.table(CUSTOM_ROLES_TABLE).collect()}
    # Canonical city names; the API validates them on save, this keeps a stray value out of the searches
    city_names = {
        row["city"].lower(): row["city"]
        for row in spark.table(INDIA_REF_TABLE).where("city IS NOT NULL AND lower(city) <> 'india'").select("city").distinct().collect()
    }

    def search_role(role):
        if role in valid_roles:
            return role
        entry = custom.get((role or "").lower())
        if entry is None:
            return None
        return {"active": entry["role_title"], "mapped": entry["duplicate_of"]}.get(entry["status"])

    def search_cities(values):
        cities = (city_names.get((value or "").strip().lower()) for value in values or [])
        return list(dict.fromkeys(city for city in cities if city))[:MAX_CITIES]

    profiles = []
    for row in (spark.table(PROFILE_TABLE).where("profile_id <> 'default'")
                .select("profile_id", "target_roles", "preferred_cities").collect()):
        roles = list(dict.fromkeys(role for role in map(search_role, row["target_roles"] or []) if role))
        if roles:
            profiles.append({"user_id": row["profile_id"], "roles": roles, "cities": search_cities(row["preferred_cities"])})

    # Ids silver keeps at least until the next daily retention (its age cutoff is DELETE_DAYS), so a re-sighting
    # without a description always finds the stored one
    seen_ids = [row["job_id"] for row in spark.sql(f"""
        SELECT DISTINCT job_id FROM {SILVER_TABLE}
        WHERE ats_type = 'linkedin' AND job_id IS NOT NULL AND nullif(trim(description), '') IS NOT NULL
          AND first_seen_at >= current_timestamp() - INTERVAL {SEEN_IDS_DAYS} DAYS
          AND {JOB_AGE_DATE} >= date_sub(current_date(), {DELETE_DAYS - 1})
        ORDER BY job_id
    """).collect()]

    generated_at = datetime.now(timezone.utc).isoformat()
    dbutils.fs.mkdirs(CONFIG_DIR)
    put_json(LINKEDIN_ROLES_FILE, {"generated_at": generated_at, "profiles": profiles}, indent=1)
    put_json(LINKEDIN_SEEN_IDS_FILE, {"generated_at": generated_at, "job_ids": seen_ids})

    distinct_roles = sorted({role for profile in profiles for role in profile["roles"]})
    distinct_cities = sorted({city for profile in profiles for city in profile["cities"]})
    metrics.update(rows_in=len(profiles), rows_out=len(distinct_roles), rows_rejected=0)
    metrics["details"].update(roles_file=LINKEDIN_ROLES_FILE, roles=distinct_roles, cities=distinct_cities,
                              seen_ids_file=LINKEDIN_SEEN_IDS_FILE, seen_ids=len(seen_ids))
    print(f"{LINKEDIN_ROLES_FILE}: {len(profiles)} profiles, roles={distinct_roles}, cities={distinct_cities}")
    print(f"{LINKEDIN_SEEN_IDS_FILE}: {len(seen_ids)} job ids")

## 2. MERGE into gold.jobs

In [ ]:
# Change detection: a row is rewritten only when one of these differs
CONTENT_COLUMNS = [
    "source", "company_name", "job_id", "title", "location", "posted_date", "job_url", "description",
    "seniority_level", "employment_type", "job_function", "industries",
    "experience_min_years", "experience_max_years", "experience_level", "skills", "skill_groups", "category",
    "role_title", "role_score", "role_alternative", "role_method", "description_hash",
]
# Refreshed with the content, but a new sighting alone does not rewrite the row
SIGHTING_COLUMNS = ["first_seen_at", "last_seen_at", "times_seen"]
PIPELINE_COLUMNS = ["job_key", *CONTENT_COLUMNS, *SIGHTING_COLUMNS]
ENRICHED_COLUMNS = {"experience_min_years", "experience_max_years", "experience_level", "skills", "skill_groups", "category",
                    "role_title", "role_score", "role_alternative", "role_method"}
# Set only on INSERT, never overwritten by the MERGE: user tracking + lifecycle (lifecycle is managed by the cleanup below)
INSERT_DEFAULTS = {
    "is_applied": "false",
    "applied_at": "CAST(NULL AS TIMESTAMP)",
    "is_hidden": "false",
    "hidden_at": "CAST(NULL AS TIMESTAMP)",
    "application_status": "'not_applied'",
    "status_updated_at": "CAST(NULL AS TIMESTAMP)",
    "is_active": "true",
    "expired_at": "CAST(NULL AS TIMESTAMP)",
}

with task_run("gold_merge") as metrics:
    source_columns = ", ".join(f"{'e' if column in ENRICHED_COLUMNS else 's'}.{column}" for column in PIPELINE_COLUMNS)
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW gold_updates AS
        SELECT {source_columns}
        FROM {SILVER_TABLE} s
        JOIN {ENRICHED_TABLE} e
          ON e.job_key = s.job_key AND e.description_hash = s.description_hash
        WHERE e.gold_eligible
          AND coalesce(s.posted_date, to_date(s.first_seen_at)) >= date_sub(current_date(), {DELETE_DAYS})
    """)

    # One action for all source checks
    check = spark.sql("""
        SELECT count(*) AS eligible, count(*) - count(DISTINCT job_key) AS duplicate_keys,
               count_if(role_title IS NULL) AS without_role
        FROM gold_updates
    """).first()
    eligible, without_role = int(check["eligible"]), int(check["without_role"])
    if check["duplicate_keys"]:
        raise ValueError(f"{check['duplicate_keys']} duplicate job_key values in the gold source; silver/enriched must be unique per job_key")

    inserted = updated = 0
    if eligible:
        changed = " OR ".join(f"NOT (t.{column} <=> s.{column})" for column in CONTENT_COLUMNS)
        update_set = ", ".join(f"t.{column} = s.{column}" for column in CONTENT_COLUMNS + SIGHTING_COLUMNS)
        insert_columns = PIPELINE_COLUMNS + list(INSERT_DEFAULTS) + ["gold_inserted_at", "gold_updated_at", "_job_run_id"]
        insert_values = (
            [f"s.{column}" for column in PIPELINE_COLUMNS]
            + list(INSERT_DEFAULTS.values())
            + ["current_timestamp()", "current_timestamp()", f"'{JOB_RUN_ID}'"]
        )
        merge_result = spark.sql(f"""
            MERGE INTO {GOLD_TABLE} t
            USING gold_updates s
            ON t.job_key = s.job_key
            WHEN MATCHED AND ({changed}) THEN UPDATE SET
              {update_set},
              t.gold_updated_at = current_timestamp(),
              t._job_run_id = '{JOB_RUN_ID}'
            WHEN NOT MATCHED THEN INSERT ({", ".join(insert_columns)})
              VALUES ({", ".join(insert_values)})
        """)
        inserted, updated = dml_rows(merge_result, "num_inserted_rows") or 0, dml_rows(merge_result, "num_updated_rows") or 0

    metrics.update(rows_in=eligible, rows_out=inserted + updated, rows_rejected=0,
                   message=None if eligible else "no eligible jobs")
    metrics["details"].update(inserted=inserted, updated=updated, unchanged=eligible - inserted - updated, without_role=without_role)
    print(f"eligible={eligible} inserted={inserted} updated={updated} unchanged={eligible - inserted - updated} "
          f"(without a role yet: {without_role} - 03_enrich assigns them in later runs)")

## 3. Cleanup

In [ ]:
with task_run("cleanup") as metrics:
    now = datetime.now(timezone.utc)

    # Gates for the daily retention and the weekly maintenance, from earlier cleanup runs (one query).
    # Cleanup runs from before retention_ran existed ran the retention every time unless they were dry runs.
    gates = spark.sql(f"""
        SELECT
          max(CASE WHEN coalesce(d.retention_ran, NOT coalesce(d.dry_run, false)) THEN started_at END) AS last_retention,
          max(CASE WHEN d.maintenance_ran THEN started_at END) AS last_maintenance
        FROM (
          SELECT started_at, from_json(details, 'struct<retention_ran:boolean,maintenance_ran:boolean,dry_run:boolean>') AS d
          FROM {RUNS_TABLE}
          WHERE task = 'cleanup' AND status = 'SUCCESS'
        )
    """).first()

    def due(last_run, hours):
        return last_run is None or now - last_run.replace(tzinfo=timezone.utc) >= timedelta(hours=hours)

    retention_due = due(gates["last_retention"], RETENTION_HOURS)
    maintenance_due = due(gates["last_maintenance"], MAINTENANCE_HOURS)

    # 3a. Every run: expire / re-activate gold jobs by posting age
    update_where(
        "gold_expired", GOLD_TABLE, "is_active = false, expired_at = current_timestamp()",
        f"coalesce(is_active, true) AND {JOB_AGE_DATE} < date_sub(current_date(), {EXPIRE_DAYS})",
    )
    update_where(
        "gold_reactivated", GOLD_TABLE, "is_active = true, expired_at = NULL",
        f"(is_active IS NULL OR NOT is_active) AND {JOB_AGE_DATE} >= date_sub(current_date(), {EXPIRE_DAYS})",
    )
    # Old expired jobs nobody tracks (applied / saved / interviewing ... by any user are kept forever)
    delete_where(
        "gold_deleted", GOLD_TABLE,
        f"NOT coalesce(is_active, true) AND {JOB_AGE_DATE} < date_sub(current_date(), {DELETE_DAYS}) AND {UNTRACKED_JOB}",
    )
    # Jobs 03_enrich excluded as outside the supported roles (kept if any user tracks them)
    delete_where(
        "gold_out_of_scope", GOLD_TABLE,
        f"EXISTS (SELECT 1 FROM {ENRICHED_TABLE} AS e WHERE e.job_key = t.job_key AND NOT e.gold_eligible "
        f"AND e.gold_exclusion_reason LIKE '%out_of_scope%') AND {UNTRACKED_JOB}",
    )
    delete_where("user_state_orphans", USER_STATE_TABLE, missing_from(GOLD_TABLE))

    # 3b. Every run: skill rarity over the active jobs (skills and skill groups, case-insensitive), read by the API
    stats_result = spark.sql(f"""
        CREATE OR REPLACE TABLE {SKILL_STATS_TABLE}
        COMMENT 'Rebuilt by 04_gold_cleanup on every run from active gold.jobs: lower(skill or skill group), jobs, idf = ln(1 + active jobs / jobs)'
        AS
        WITH active AS (
          SELECT job_key, skills, skill_groups FROM {GOLD_TABLE} WHERE coalesce(is_active, true)
        ),
        job_skill AS (
          SELECT DISTINCT job_key, lower(trim(skill)) AS skill
          FROM (
            SELECT job_key, explode(array_distinct(array_union(
              coalesce(skills, CAST(array() AS ARRAY<STRING>)), coalesce(skill_groups, CAST(array() AS ARRAY<STRING>))
            ))) AS skill
            FROM active
          )
          WHERE nullif(trim(skill), '') IS NOT NULL
        )
        SELECT js.skill, CAST(count(*) AS BIGINT) AS jobs, CAST(ln(1 + n.jobs / count(*)) AS DOUBLE) AS idf
        FROM job_skill js
        CROSS JOIN (SELECT count(*) AS jobs FROM active) n
        GROUP BY js.skill, n.jobs
    """)
    skill_stats_rows = dml_rows(stats_result, "num_inserted_rows")
    print(f"{SKILL_STATS_TABLE}: rebuilt ({skill_stats_rows if skill_stats_rows is not None else '?'} skills)")

    # 3c. Retention, at most every RETENTION_HOURS (a dry run always reports it)
    removed_files = []
    run_retention = retention_due or DRY_RUN
    if not run_retention:
        print(f"retention skipped (last={gates['last_retention']}, interval={RETENTION_HOURS}h)")
    else:
        # Landing files already ingested by bronze and older than the retention
        ingested = {
            row["_source_file"].replace("dbfs:", "")
            for row in spark.table(BRONZE_TABLE).select("_source_file").distinct().collect()
            if row["_source_file"]
        }
        cutoff_ms = int((now - timedelta(days=LANDING_DAYS)).timestamp() * 1000)

        def walk(path):
            for info in dbutils.fs.ls(path):
                if info.isDir():
                    yield from walk(info.path)
                else:
                    yield info

        kept_not_ingested = 0
        for info in walk(LANDING_PATH):
            file_path = info.path.replace("dbfs:", "")
            if file_path not in ingested:
                kept_not_ingested += 1
                continue
            if info.modificationTime < cutoff_ms:
                if not DRY_RUN:
                    dbutils.fs.rm(info.path)
                removed_files.append(file_path)

        removed_folders = 0
        if not DRY_RUN:
            for folder in dbutils.fs.ls(LANDING_PATH):
                if folder.isDir() and not dbutils.fs.ls(folder.path):
                    dbutils.fs.rm(folder.path, recurse=True)
                    removed_folders += 1
        report["landing_files_deleted"] = len(removed_files)
        report["landing_folders_deleted"] = removed_folders
        report["landing_files_not_ingested"] = kept_not_ingested
        print(f"{'[DRY RUN] would delete' if DRY_RUN else 'deleted'} {len(removed_files):>6} landing files "
              f"({removed_folders} empty folders, {kept_not_ingested} not yet ingested kept)")

        # Bronze rows from runs silver has processed
        spark.sql(f"""
            SELECT DISTINCT explode(from_json(details, 'struct<bronze_run_ids:array<string>>').bronze_run_ids) AS run_id
            FROM {RUNS_TABLE}
            WHERE task = 'silver_transform' AND status = 'SUCCESS'
        """).createOrReplaceTempView("processed_bronze_runs")
        delete_where(
            "bronze_rows", BRONZE_TABLE,
            f"_ingested_at < current_timestamp() - INTERVAL {BRONZE_DAYS} DAYS "
            f"AND _job_run_id IN (SELECT run_id FROM processed_bronze_runs)",
        )

        # Rejected rows and run log
        delete_where("rejected_rows", REJECTED_TABLE, f"_rejected_at < current_timestamp() - INTERVAL {REJECTED_DAYS} DAYS")
        delete_where("run_log_rows", RUNS_TABLE, f"started_at < current_timestamp() - INTERVAL {RUN_LOG_DAYS} DAYS")

        # Old silver rows that are not in gold (the gold source has the same age cutoff, so they would not be merged)
        delete_where(
            "silver_rows", SILVER_TABLE,
            f"{JOB_AGE_DATE} < date_sub(current_date(), {DELETE_DAYS}) AND {missing_from(GOLD_TABLE)}",
        )
        delete_where("enriched_orphans", ENRICHED_TABLE, missing_from(SILVER_TABLE))

        # LLM log: failures, legacy rows, superseded prompt versions, orphans.
        # Prompt family = first letter of prompt_version (p = skills, r = roles); the latest version per endpoint + family is kept.
        delete_where(
            "llm_failed_or_legacy", LLM_LOG_TABLE,
            f"(error IS NOT NULL OR response IS NULL OR prompt_version IS NULL) "
            f"AND called_at < current_timestamp() - INTERVAL {LLM_FAILED_DAYS} DAYS",
        )
        current_prompts = (
            spark.table(LLM_LOG_TABLE).where("prompt_version IS NOT NULL")
            .groupBy("endpoint", F.substring("prompt_version", 1, 1).alias("family"))
            .agg(F.max_by("prompt_version", "called_at").alias("prompt_version"))
            .collect()
        )
        if current_prompts:
            keep = " OR ".join(
                f"(endpoint = {sql_literal(row['endpoint'])} AND prompt_version = {sql_literal(row['prompt_version'])})"
                for row in current_prompts
            )
            delete_where(
                "llm_old_prompt_versions", LLM_LOG_TABLE,
                f"prompt_version IS NOT NULL AND NOT ({keep}) AND called_at < current_timestamp() - INTERVAL 30 DAYS",
            )
        delete_where(
            "llm_orphans", LLM_LOG_TABLE,
            f"{missing_from(SILVER_TABLE)} AND called_at < current_timestamp() - INTERVAL {LLM_FAILED_DAYS} DAYS",
        )

        # Embedding cache: failed calls and embeddings of jobs no longer in silver. Only job keys (sha2 hex) are pruned,
        # so role embeddings ('role:' keys) and any other non-job key are kept.
        delete_where(
            "embedding_failed", EMBEDDING_CACHE_TABLE,
            f"embedding IS NULL AND created_at < current_timestamp() - INTERVAL {LLM_FAILED_DAYS} DAYS",
        )
        delete_where(
            "embedding_orphans", EMBEDDING_CACHE_TABLE,
            f"cache_key RLIKE '^[0-9a-f]{{64}}$' AND {missing_from(SILVER_TABLE, column='cache_key')} "
            f"AND created_at < current_timestamp() - INTERVAL {LLM_FAILED_DAYS} DAYS",
        )

    # 3d. Delta maintenance, at most every MAINTENANCE_HOURS
    maintenance_errors = {}
    if maintenance_due and not DRY_RUN:
        for table_name in MAINTAINED_TABLES:
            try:
                spark.sql(f"OPTIMIZE {table_name}")
                spark.sql(f"VACUUM {table_name}")
                print(f"maintained {table_name}")
            except Exception as exc:  # maintenance must never fail the pipeline
                maintenance_errors[table_name] = str(exc).splitlines()[0][:300]
                print(f"[WARN] maintenance failed for {table_name}: {maintenance_errors[table_name]}")
    else:
        print(f"maintenance skipped (due={maintenance_due}, last={gates['last_maintenance']}, dry_run={DRY_RUN})")

    total_removed = sum(value for key, value in report.items() if key not in ("landing_files_not_ingested", "gold_reactivated"))
    metrics.update(rows_in=None, rows_out=total_removed, rows_rejected=0, files_read=len(removed_files),
                   message="dry run" if DRY_RUN else None)
    metrics["details"].update(
        report, dry_run=DRY_RUN, skill_stats_rows=skill_stats_rows,
        retention_ran=bool(run_retention and not DRY_RUN), maintenance_ran=bool(maintenance_due and not DRY_RUN),
        maintenance_errors=maintenance_errors,
    )

## 4. Publish files for Supabase
Written with Python file I/O on the volume (gzip for the jobs), `manifest.json` last. Session time zone UTC for the timestamp strings. Runs in a dry run too (gold is merged then as well).

In [ ]:
import gzip
import hashlib
import math
import os
import re
import time

PUBLISH_DIR = f"{PIPELINE_VOLUME}/publish"
PUBLISH_RUN_DIR = f"{PUBLISH_DIR}/{JOB_RUN_ID}"
PUBLISH_FORMAT = "jobseeker.publish.v1"
PUBLISH_ROWS_PER_FILE = 2000
PUBLISH_KEEP_COMPLETE = 3       # newest complete directories always kept (this run's included)
PUBLISH_COMPLETE_DAYS = 3       # older complete directories beyond those are deleted after N days
PUBLISH_INCOMPLETE_DAYS = 1     # directories without manifest.json (a failed run) are deleted after N days
CUSTOM_SKILLS_TABLE = table("ops", "custom_skills")
REF_SKILLS_TABLE = table("ops", "ref_skills")
ROLE_SIMILARITY_TABLE = table("ops", "role_similarity")



def iso_utc(column):
    """SQL for a timestamp as 'YYYY-MM-DDTHH:MM:SS.ffffffZ' (the session time zone is set to UTC below); NULL stays NULL."""
    return f"concat(date_format({column}, 'yyyy-MM-dd'), 'T', date_format({column}, 'HH:mm:ss.SSSSSS'), 'Z')"


# app.jobs content columns (Supabase schema app, supabase/app_schema.sql); timestamps as UTC ISO strings, dates as YYYY-MM-DD
PUBLISH_JOB_COLUMNS = {
    "job_key": "job_key", "source": "source", "company_name": "company_name", "title": "title", "location": "location",
    "posted_date": "CAST(posted_date AS STRING)", "job_url": "job_url", "seniority_level": "seniority_level",
    "employment_type": "employment_type", "experience_min_years": "experience_min_years",
    "experience_max_years": "experience_max_years", "experience_level": "experience_level", "skills": "skills",
    "skill_groups": "skill_groups", "category": "category", "role_title": "role_title", "role_score": "role_score",
    "role_alternative": "role_alternative", "role_method": "role_method",
    "first_seen_at": iso_utc("first_seen_at"), "last_seen_at": iso_utc("last_seen_at"),
    "times_seen": "times_seen", "is_active": "is_active", "description": "description", "job_function": "job_function",
    "industries": "industries",
}


def publish_value(value):
    """JSON-safe value: NaN / Infinity -> None, arrays element-wise."""
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, (list, tuple)):
        return [publish_value(item) for item in value]
    return value


def canonical_json(payload):
    return json.dumps(payload, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)


def publish_row_hash(row):
    """sha256 of the canonical JSON of the row without "h" (publish_to_supabase.py recomputes it the same way)."""
    return hashlib.sha256(canonical_json({k: v for k, v in row.items() if k != "h"}).encode("utf-8")).hexdigest()


def publish_rows(query, key_columns, label):
    """Rows of a small table as JSON-safe dicts, first row per key kept (the Supabase tables have that key as PK)."""
    rows, seen, duplicates = [], set(), 0
    for row in spark.sql(query).collect():
        record = {name: publish_value(value) for name, value in row.asDict().items()}
        key = tuple(record[column] for column in key_columns)
        if key in seen:
            duplicates += 1
            continue
        seen.add(key)
        rows.append(record)
    if duplicates:
        print(f"[WARN] {label}: {duplicates} duplicate keys skipped")
        report_publish["duplicates"][label] = duplicates
    return rows


report_publish = {"duplicates": {}}

with task_run("publish_files") as metrics:
    spark.conf.set("spark.sql.session.timeZone", "UTC")
    started_all = time.monotonic()
    snapshot_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ")
    timings = {}

    # A repaired run rewrites its directory: drop the old manifest first so the directory is never complete half-written
    os.makedirs(PUBLISH_RUN_DIR, exist_ok=True)
    if os.path.exists(f"{PUBLISH_RUN_DIR}/manifest.json"):
        os.remove(f"{PUBLISH_RUN_DIR}/manifest.json")
    for name in os.listdir(PUBLISH_RUN_DIR):
        if os.path.isfile(f"{PUBLISH_RUN_DIR}/{name}"):
            os.remove(f"{PUBLISH_RUN_DIR}/{name}")

    file_sha, file_bytes = {}, {}

    def write_file(name, data):
        with open(f"{PUBLISH_RUN_DIR}/{name}", "wb") as handle:
            handle.write(data)
        file_sha[name] = hashlib.sha256(data).hexdigest()
        file_bytes[name] = len(data)

    def write_json(name, payload):
        write_file(name, json.dumps(payload, ensure_ascii=False, separators=(",", ":"), allow_nan=False).encode("utf-8"))

    # Jobs: the full gold snapshot, sorted by job_key, PUBLISH_ROWS_PER_FILE rows per gzip file
    started = time.monotonic()
    job_files, keys, part = [], [], []
    invalid_keys = duplicate_keys = 0
    published_keys = set()  # a set, not the sort order: a duplicate key would make the publisher refuse the whole run

    def flush_jobs():
        name = f"jobs-{len(job_files):05d}.jsonl.gz"
        write_file(name, gzip.compress(("\n".join(part) + "\n").encode("utf-8"), compresslevel=6, mtime=0))
        job_files.append(name)
        part.clear()

    select_jobs = ", ".join(f"{expression} AS {name}" for name, expression in PUBLISH_JOB_COLUMNS.items())
    for row in spark.sql(f"SELECT {select_jobs} FROM {GOLD_TABLE} ORDER BY job_key").toLocalIterator():
        record = {name: publish_value(row[name]) for name in PUBLISH_JOB_COLUMNS}
        key = record["job_key"]
        if not isinstance(key, str) or not re.fullmatch(r"[0-9a-f]{64}", key):
            invalid_keys += 1
            continue
        if key in published_keys:
            duplicate_keys += 1
            continue
        published_keys.add(key)
        record["h"] = publish_row_hash(record)
        keys.append([key, record["h"]])
        part.append(canonical_json(record))
        if len(part) >= PUBLISH_ROWS_PER_FILE:
            flush_jobs()
    if part:
        flush_jobs()
    write_json("keys.json", {"run_id": JOB_RUN_ID, "keys": keys})
    timings["jobs"] = round(time.monotonic() - started, 1)

    # Small tables, each as its own file
    started = time.monotonic()
    skill_stats = publish_rows(
        f"SELECT skill, jobs, idf FROM {SKILL_STATS_TABLE} WHERE skill IS NOT NULL ORDER BY skill", ["skill"], "skill_stats")
    role_similarity = publish_rows(
        f"SELECT role_a, role_b, sim FROM {ROLE_SIMILARITY_TABLE} WHERE role_a IS NOT NULL AND role_b IS NOT NULL "
        "ORDER BY role_a, role_b, sim DESC", ["role_a", "role_b"], "role_similarity")
    refs = {
        "ref_roles": publish_rows(
            f"SELECT role_title, category, coalesce(in_scope, false) AS in_scope FROM {REF_ROLES_TABLE} "
            "WHERE role_title IS NOT NULL ORDER BY role_title", ["role_title"], "ref_roles"),
        "ref_skills": publish_rows(
            f"SELECT skill, skill_group, aliases FROM {REF_SKILLS_TABLE} WHERE skill IS NOT NULL ORDER BY skill",
            ["skill"], "ref_skills"),
        "ref_cities": publish_rows(
            f"SELECT alias, city, state FROM {INDIA_REF_TABLE} WHERE alias IS NOT NULL ORDER BY alias, city DESC NULLS LAST, state DESC",
            ["alias"], "ref_cities"),  # first row per alias = max(city), as the old CITY_NAMES map
    }
    custom_statuses = {
        "custom_roles": publish_rows(f"""
            SELECT role_title, status, category, description, duplicate_of, first_user, checked_by,
                   {iso_utc("created_at")} AS created_at, {iso_utc("checked_at")} AS checked_at, attempts
            FROM {CUSTOM_ROLES_TABLE} WHERE role_title IS NOT NULL
              AND status IN ('pending', 'active', 'mapped', 'rejected')
            ORDER BY lower(role_title), checked_at DESC NULLS LAST""", ["role_title"], "custom_roles"),
        "custom_skills": publish_rows(f"""
            SELECT skill, first_user, {iso_utc("created_at")} AS created_at
            FROM {CUSTOM_SKILLS_TABLE} WHERE skill IS NOT NULL ORDER BY lower(skill), created_at""", ["skill"],
            "custom_skills"),
    }
    # Supabase keys custom roles / skills on lower(): keep the first row per lower-case name
    for kind, column in (("custom_roles", "role_title"), ("custom_skills", "skill")):
        seen, kept = set(), []
        for record in custom_statuses[kind]:
            if record[column].lower() not in seen:
                seen.add(record[column].lower())
                kept.append(record)
        if len(kept) != len(custom_statuses[kind]):
            report_publish["duplicates"][kind] = len(custom_statuses[kind]) - len(kept)
        custom_statuses[kind] = kept

    write_json("skill_stats.json", {"rows": skill_stats})
    write_json("role_similarity.json", {"rows": role_similarity})
    write_json("refs.json", refs)
    write_json("custom_statuses.json", custom_statuses)
    timings["small_files"] = round(time.monotonic() - started, 1)

    loaded = spark.sql(f"""
        SELECT get_json_object(details, '$.exported_at') AS exported_at FROM {RUNS_TABLE}
        WHERE job_run_id = '{JOB_RUN_ID}' AND task = 'load_user_data' AND status = 'SUCCESS'
          AND get_json_object(details, '$.loaded') = 'true'
        ORDER BY started_at DESC LIMIT 1
    """).first()
    counts = {
        "jobs": len(keys), "skill_stats": len(skill_stats), "role_similarity": len(role_similarity),
        **{kind: len(rows) for kind, rows in refs.items()},
        **{kind: len(rows) for kind, rows in custom_statuses.items()},
    }
    manifest = {
        "format": PUBLISH_FORMAT, "run_id": JOB_RUN_ID, "catalog": CATALOG, "snapshot_at": snapshot_at,
        "user_data_exported_at": loaded["exported_at"] if loaded else None,
        "files": {"jobs": job_files, "keys": "keys.json", "skill_stats": "skill_stats.json",
                  "role_similarity": "role_similarity.json", "refs": "refs.json", "custom_statuses": "custom_statuses.json"},
        "sha256": dict(file_sha), "bytes": dict(file_bytes), "counts": counts,
    }
    # Written last: its presence marks the directory complete for publish_to_supabase.py
    with open(f"{PUBLISH_RUN_DIR}/manifest.json", "w", encoding="utf-8") as handle:
        handle.write(json.dumps(manifest, ensure_ascii=False, indent=1))

    # Retention of older publish directories (never this run's); a problem here never fails the publish
    started = time.monotonic()
    retention = {"deleted_complete": [], "deleted_incomplete": [], "kept": 0, "errors": []}
    try:
        now_seconds = time.time()
        complete, incomplete = [], []
        for name in os.listdir(PUBLISH_DIR):
            path = f"{PUBLISH_DIR}/{name}"
            if name == JOB_RUN_ID or not os.path.isdir(path):
                continue
            if os.path.exists(f"{path}/manifest.json"):
                try:
                    with open(f"{path}/manifest.json", encoding="utf-8") as handle:
                        snapshot = str(json.load(handle).get("snapshot_at") or "")
                except Exception:
                    snapshot = ""
                complete.append((snapshot, os.path.getmtime(f"{path}/manifest.json"), name))
            else:
                mtimes = [os.path.getmtime(f"{path}/{child}") for child in os.listdir(path)] or [os.path.getmtime(path)]
                incomplete.append((max(mtimes), name))
        complete.sort(reverse=True)  # newest snapshot first (same timestamp format everywhere)
        to_delete = [name for _, mtime, name in complete[PUBLISH_KEEP_COMPLETE - 1:]
                     if now_seconds - mtime > PUBLISH_COMPLETE_DAYS * 86400]
        to_delete_incomplete = [name for mtime, name in incomplete if now_seconds - mtime > PUBLISH_INCOMPLETE_DAYS * 86400]
        for name in to_delete + to_delete_incomplete:
            if not DRY_RUN:
                dbutils.fs.rm(f"{PUBLISH_DIR}/{name}", recurse=True)
        retention.update(deleted_complete=to_delete, deleted_incomplete=to_delete_incomplete,
                         kept=len(complete) - len(to_delete) + 1)
    except Exception as exc:
        retention["errors"].append(f"{type(exc).__name__}: {str(exc)[:300]}")
        print(f"[WARN] publish retention failed: {retention['errors'][-1]}")
    timings["retention"] = round(time.monotonic() - started, 1)
    timings["total"] = round(time.monotonic() - started_all, 1)

    total_bytes = sum(file_bytes.values())
    metrics.update(rows_in=len(keys) + invalid_keys + duplicate_keys, rows_out=len(keys),
                   rows_rejected=invalid_keys + duplicate_keys, files_read=None,
                   message="dry run (files written, retention reported only)" if DRY_RUN else None)
    metrics["details"].update(
        dir=PUBLISH_RUN_DIR, snapshot_at=snapshot_at, counts=counts, files=len(file_sha) + 1, job_files=len(job_files),
        total_bytes=total_bytes, invalid_keys=invalid_keys, duplicate_keys=duplicate_keys,
        duplicates=report_publish["duplicates"], user_data_exported_at=manifest["user_data_exported_at"],
        retention=retention, dry_run=DRY_RUN, seconds=timings,
    )
    print(f"{PUBLISH_RUN_DIR}: {counts['jobs']} jobs in {len(job_files)} files, {total_bytes / 1e6:.1f} MB, counts={counts}")
    print(f"{'[DRY RUN] would delete' if DRY_RUN else 'deleted'} publish dirs: "
          f"{len(retention['deleted_complete'])} complete, {len(retention['deleted_incomplete'])} incomplete; {timings}")

In [ ]:
stop_if_job()  # scheduled runs end here; the cells below are for manual inspection

## Inspect

In [ ]:
display(spark.createDataFrame([(step, int(value)) for step, value in report.items()], "step STRING, rows INT"))

In [ ]:
gold = spark.table(GOLD_TABLE)
display(
    gold.groupBy("source", F.coalesce("is_active", F.lit(True)).alias("is_active"))
    .agg(
        F.count("*").alias("jobs"),
        F.round(F.avg(F.size("skills")), 1).alias("avg_skills"),
        F.sum(F.col("experience_min_years").isNotNull().cast("int")).alias("with_experience"),
        F.sum(F.col("role_title").isNull().cast("int")).alias("without_role"),
        F.max("last_seen_at").alias("latest_seen"),
    )
    .orderBy("source", "is_active")
)

In [ ]:
display(gold.groupBy("category", "role_title", "role_method").count().orderBy(F.col("count").desc()).limit(50))

In [ ]:
display(spark.table(SKILL_STATS_TABLE).orderBy(F.col("jobs").desc()).limit(30))

In [ ]:
print(dbutils.fs.head(LINKEDIN_ROLES_FILE, 4000))
print(dbutils.fs.head(LINKEDIN_SEEN_IDS_FILE, 500))

In [ ]:
with open(f"{PUBLISH_RUN_DIR}/manifest.json", encoding="utf-8") as handle:
    print(handle.read()[:4000])
display(spark.createDataFrame([(name,) for name in sorted(os.listdir(PUBLISH_DIR))], "publish_dir STRING"))

In [ ]:
display(
    spark.table(RUNS_TABLE).where("task IN ('export_linkedin_config', 'gold_merge', 'cleanup', 'publish_files')")
    .orderBy(F.col("started_at").desc()).limit(10)
)